In [1]:
from model.Multi_Scale_Patch_Mixer_with_SE import MultiscaleMixer

model = MultiscaleMixer(3, 768, 4, 0.1, [[224, 2], [224, 4]])

In [2]:
model

MultiscaleMixer(
  (patch_embedding): ModuleList(
    (0): Conv2d(3, 768, kernel_size=(224, 2), stride=(224, 2))
    (1): Conv2d(3, 768, kernel_size=(224, 4), stride=(224, 4))
  )
  (positional_embedding): ModuleList(
    (0-1): 2 x PositionalEmbedding()
  )
  (channel_mixer): ModuleList(
    (0-1): 2 x ModuleList(
      (0-3): 4 x MlpBlock(
        (mlp): Sequential(
          (0): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=768, out_features=1536, bias=True)
          (2): GELU(approximate='none')
          (3): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
          (4): Linear(in_features=1536, out_features=768, bias=True)
          (5): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (inter_mixer): ModuleList(
    (0): ModuleList(
      (0-3): 4 x MlpBlock(
        (mlp): Sequential(
          (0): LayerNorm((112,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=112, out_features=224, bias=True

In [5]:
model.channel_mixer[0][0]

MlpBlock(
  (mlp): Sequential(
    (0): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=768, out_features=1536, bias=True)
    (2): GELU(approximate='none')
    (3): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
    (4): Linear(in_features=1536, out_features=768, bias=True)
    (5): Dropout(p=0.1, inplace=False)
  )
)

In [2]:
import numpy as np

x = np.load('./data/IAA_DWT_Sobel_3/train/Drinking/x5P01A05R03_00003_DWT_3_1.npy')

In [3]:
x.shape

(4, 224, 224)

In [3]:
from model.MSPS_Mixer import MultiscaleMixer
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MultiscaleMixer().to(device=device)
checkpoint_path = './checkpoints/MSPS_Mixer_128_22_20251029_173655.pth'

model.load_state_dict(torch.load(checkpoint_path))
model.eval()

MultiscaleMixer(
  (patch_embedding): ModuleList(
    (0): Conv2d(3, 128, kernel_size=(224, 2), stride=(224, 2))
    (1): Conv2d(3, 128, kernel_size=(224, 4), stride=(224, 4))
  )
  (positional_embedding): ModuleList(
    (0-1): 2 x PositionalEmbedding()
  )
  (blocks): ModuleList(
    (0): ModuleList(
      (0): BasicLayer(
        (Shift): ModuleList(
          (0-1): 2 x Sequential(
            (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (1): ShiftBlock(
              (channel_mixer_S): Sequential(
                (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
                (1): MlpBlock(
                  (mlp): Sequential(
                    (0): Linear(in_features=128, out_features=128, bias=True)
                    (1): ReLU()
                    (2): Dropout(p=0.1, inplace=False)
                  )
                )
              )
              (channel_projection): Sequential(
                (0): MlpBlock(
                  (mlp): 

In [4]:
import torchvision
from tqdm import tqdm

test_file = './data/STFT/test'
transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
dataset = torchvision.datasets.ImageFolder(root=test_file, transform=transform)
loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False)

all_predictions = []
all_targets = []
correct = 0
total = 0

with torch.inference_mode():
    for x, y in tqdm(loader, desc="Testing"):
        if y.ndim == 2:
            y = torch.argmax(y, dim=1)
        
        x, y = x.to(device), y.to(device)
        
        logits, z = model(x)
        predictions = logits.argmax(1)
        
        all_predictions.extend(predictions.cpu().numpy())
        all_targets.extend(y.cpu().numpy())
        
        correct += (predictions == y).sum().item()
        total += y.size(0)

test_accuracy = correct / total

Testing: 100%|██████████| 6/6 [00:10<00:00,  1.77s/it]


In [5]:
test_accuracy

0.856353591160221